# 06 · Function Tools: Live Data from data.gov.sg 🌦️

A **function tool** is a normal Python function that the AI can ask us to run.

```
You:   "Is it raining in Tampines?"
AI:    "Please run get_weather('Tampines')"      <- a function_call
We:    run it, send back "Tampines: Cloudy"      <- a function_call_output
AI:    "No rain right now - just cloudy!"
```
The AI never runs your code itself — **you** run it and send back the answer.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. A Python function that calls data.gov.sg
data.gov.sg gives a free 2-hour forecast for every area in Singapore — no key needed.

In [ ]:
import requests

data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast").json()
data["data"]["items"][0]["forecasts"][:3]

In [ ]:
def get_weather(area):
    data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast").json()
    for forecast in data["data"]["items"][0]["forecasts"]:
        if area.lower() in forecast["area"].lower():
            return f"{forecast['area']}: {forecast['forecast']}"
    return f"No area called {area}"


print(get_weather("Tampines"))

## 2. Describe the tool for the AI
The AI can't see your code. It only sees a **name**, a **description** and the **inputs** — so write them clearly!

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the live 2-hour weather forecast for a Singapore area, e.g. 'Bedok' or 'Tampines'.",
        "parameters": {
            "type": "object",
            "properties": {"area": {"type": "string"}},   # the input the AI must give us
            "required": ["area"],
        },
    },
]

## 3. The AI asks for the tool

In [ ]:
response = client.responses.create(model=MODEL, instructions=bot, tools=tools, input="What's the weather in Tampines right now?")

for item in response.output:
    print(item.type)

In [ ]:
call = response.output[-1]   # the function_call is the last item

print("Name:", call.name)
print("Inputs:", call.arguments)
print("Call ID:", call.call_id)

## 4. We run it and send the answer back
`arguments` is JSON text, so `json.loads` turns it into a dictionary (notebook 02!).
`previous_response_id` tells the AI *"continue from your last response"*.

In [ ]:
import json

arguments = json.loads(call.arguments)   # {"area": "Tampines"}
result = get_weather(arguments["area"])
print(result)

response = client.responses.create(
    model=MODEL,
    instructions=bot,
    tools=tools,
    previous_response_id=response.id,
    input=[{"type": "function_call_output", "call_id": call.call_id, "output": result}],
)

print(response.output_text)

## 5. A second tool: air quality (PSI) 💨

In [ ]:
def get_air_quality():
    data = requests.get("https://api-open.data.gov.sg/v2/real-time/api/psi").json()
    psi = data["data"]["items"][0]["readings"]["psi_twenty_four_hourly"]
    return f"24-hour PSI by region: {psi}"   # under 50 = good, 51-100 = moderate


print(get_air_quality())

In [ ]:
tools.append({
    "type": "function",
    "name": "get_air_quality",
    "description": "Get the live air quality (24-hour PSI) for north, south, east, west and central Singapore.",
    "parameters": {"type": "object", "properties": {}},   # no inputs needed
})

## 6. Put it together
The AI can ask for **several** tools at once, so we use a `for` loop to run each one, then send all the results back together.
`if` / `elif` picks which of our functions to run — just like the Day 1 rules!
This is the same code as part 4 of `main.py`.

In [ ]:
def ask(text):
    response = client.responses.create(model=MODEL, instructions=bot, tools=tools, input=text)

    # Did the AI ask to use our functions? Run them.
    results = []
    for item in response.output:
        if item.type == "function_call":
            arguments = json.loads(item.arguments)

            if item.name == "get_weather":                    # which of OUR functions did it ask for?
                output = get_weather(arguments["area"])
            elif item.name == "get_air_quality":
                output = get_air_quality()

            results.append({"type": "function_call_output", "call_id": item.call_id, "output": output})
            print("🔧", item.name, arguments)

    # If it did, send the results back so it can write the final reply
    if results:
        response = client.responses.create(
            model=MODEL, instructions=bot, tools=tools, previous_response_id=response.id, input=results
        )

    return response.output_text


print(ask("I want to cycle in Bedok this afternoon. Is the weather and air OK?"))

## ✏️ Exercises
1. Explore more live data at **data.gov.sg** (search "real-time"): try `air-temperature` or `uv`.
   Write a `get_temperature()` tool from `https://api-open.data.gov.sg/v2/real-time/api/air-temperature`.
2. Make `get_air_quality` say *"good"* or *"moderate"* instead of just numbers.
3. 🌶️ Challenge: a `get_time()` tool so the bot knows if it's morning or night.

In [ ]:
# your code here